# GridSpark AI – Notebook 03: Label Design & Supervised Learning Demo

This notebook explains how to frame the wildfire ignition risk problem as a supervised binary classification task, defines positive and negative samples, and documents critical constraints around spatial/temporal leakage.

> **Purpose of model:** Decision support for inspection prioritization, vegetation management, and risk screening — NOT automated infrastructure shutoffs.

## 1. Problem framing

**Question:** Given a transmission-line corridor segment at a specific date/time with measured fire-weather conditions and pre-fire vegetation state, what is the estimated probability that an ignition occurs within a buffer zone?

**Output:** A corridor-level risk score (0–1) useful for prioritizing field inspection or vegetation management crews — not a shutoff decision.

## 2. Sample design

### Positive samples (label = 1)

- A **grid corridor segment** (from HIFLD transmission lines) located within `buffer_m` meters of a known FPA-FOD ignition point.
- **Date/time window:** The 24–72 hours leading up to the FPA-FOD reported ignition date (fire-weather window).
- **Conditions:** Fire-weather variables from the nearest NOAA ISD station on that date.
- **Vegetation state:** Sentinel-2 NDVI/NDMI/NBR from the 30-day pre-fire window.

**⚠️ Important:** FPA-FOD coordinates carry positional uncertainty. A 1 km buffer is used as the minimum matching radius; 3 km and 5 km buffers are tested for sensitivity.

### Negative samples (label = 0)

- A corridor segment **in the same county and same calendar season** (June–September) in a year where **no FPA-FOD ignition** was recorded within `buffer_m` meters.
- Weather conditions must be on **comparable fire-weather days** (e.g., relative humidity < 25%, wind speed > 5 m/s) to avoid trivially easy negatives.
- Spatial separation: negative corridors should be **≥ 10 km from any positive-labeled ignition** to avoid spatial leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Illustrative synthetic schema
np.random.seed(42)
n_pos, n_neg = 30, 70

pos = pd.DataFrame({
    'ignition_label': 1,
    'buffer_m': np.random.choice([1000, 3000, 5000], n_pos),
    'ndvi_mean': np.random.uniform(0.3, 0.7, n_pos),
    'ndmi_mean': np.random.uniform(-0.3, 0.0, n_pos),   # drier
    'relative_humidity': np.random.uniform(10, 30, n_pos),  # low RH
    'wind_speed': np.random.uniform(5, 15, n_pos),
    'label_source': 'FPA-FOD',
    'split': 'test',  # Holiday Farm event = test set
})

neg = pd.DataFrame({
    'ignition_label': 0,
    'buffer_m': np.random.choice([1000, 3000, 5000], n_neg),
    'ndvi_mean': np.random.uniform(0.4, 0.8, n_neg),
    'ndmi_mean': np.random.uniform(0.0, 0.3, n_neg),    # wetter
    'relative_humidity': np.random.uniform(35, 80, n_neg),  # higher RH
    'wind_speed': np.random.uniform(0, 6, n_neg),
    'label_source': 'no_ignition',
    'split': 'train',
})

df = pd.concat([pos, neg], ignore_index=True)
print(f'Total samples: {len(df)}  |  Positive: {n_pos}  |  Negative: {n_neg}')
print(f'Class balance: {n_pos / len(df):.1%} positive')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, feat, xlabel in zip(
    axes,
    ['relative_humidity', 'ndmi_mean'],
    ['Relative Humidity (%)', 'NDMI (pre-fire)'],
):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        sub = df[df['ignition_label'] == label][feat]
        ax.hist(sub, bins=15, alpha=0.6, color=color,
                label='Ignition' if label == 1 else 'No ignition')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    ax.legend()
    ax.set_title(f'Illustrative: {xlabel} by label')

plt.suptitle('GridSpark AI — Synthetic label design illustration (not real data)', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/label_design_illustration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 3. Train / test split strategy

### ⚠️ Do NOT use random pixel-level splitting

Random splits create **spatial leakage**: nearby pixels in train and test sets share correlated features (terrain, vegetation, local climate), causing inflated test performance that does not generalize.

### Recommended split: Event-out or County-out cross-validation

| Split type | Description | Use when |
|---|---|---|
| **Event-out** | Hold out one fire event as test; train on all others | You have multiple fire events |
| **County-out** | Hold out one county; train on neighboring counties | You have multiple counties |
| **Year-out** | Hold out one fire season; train on prior years | Temporal generalization is priority |

For this prototype, **Holiday Farm Fire = test set**. Training would use other Oregon 2020 fire events or 2017–2019 fire seasons.

In [ ]:
# Illustrative event-out split
train = df[df['split'] == 'train']
test  = df[df['split'] == 'test']

print(f'Train set: {len(train)} samples  |  {train["ignition_label"].sum()} positive')
print(f'Test set:  {len(test)} samples   |  {test["ignition_label"].sum()} positive')
print('\nNote: Positive samples (Holiday Farm) are held out as test-only.')
print('This simulates generalization to unseen events.')

## 4. Temporal leakage warning

Do not use any weather observations **after the ignition time** in training features for the corresponding positive sample. The feature window must be strictly pre-ignition:

```
Feature window:  [ignition_date - 72h]  →  [ignition_date - 1h]
                  ↑                          ↑
                  OK                         OK (last pre-fire hour)
                                             
Label time:      [ignition_date 20:20 PDT]  ← Do NOT include this in features
```

## 5. Class imbalance

Wildfire ignitions near transmission corridors are rare events. Expected class imbalance: 1–5% positive.

Recommended handling:
- Use **class-weighted loss** (e.g., `class_weight='balanced'` in scikit-learn)
- Evaluate with **precision-recall AUC**, not accuracy or ROC-AUC alone
- Use **stratified folds** if doing cross-validation within the training set

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, average_precision_score
from sklearn.preprocessing import StandardScaler

features = ['ndvi_mean', 'ndmi_mean', 'relative_humidity', 'wind_speed']

X_train = train[features].values
y_train = train['ignition_label'].values
X_test  = test[features].values
y_test  = test['ignition_label'].values

# Demonstration only — not a real trained model
clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print('=== SYNTHETIC DATA ONLY — NOT A REAL MODEL ===')
print('\nClassification report (Holiday Farm test set):')
print(classification_report(y_test, y_pred, target_names=['No ignition', 'Ignition']))
print(f'Average Precision Score: {average_precision_score(y_test, y_prob):.3f}')

In [ ]:
importances = pd.Series(clf.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 3))
importances.plot.barh(ax=ax, color='steelblue')
ax.set_title('Feature importance — illustrative synthetic model only')
ax.set_xlabel('Mean decrease in impurity')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance_illustration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 6. Checklist before real model training

- [ ] All five feasibility matrix layers show `Pass`
- [ ] FPA-FOD file downloaded and filtered to OR 2020
- [ ] HIFLD transmission corridors clipped to Lane County AOI
- [ ] Sentinel-2 features exported from GEE for pre-fire window
- [ ] NOAA ISD hourly data verified for Aug–Sep 2020
- [ ] MTBS/WFIGS perimeter confirmed for Holiday Farm
- [ ] Train/test split is event-out or county-out (NOT random)
- [ ] Evaluation metrics include precision-recall AUC
- [ ] Model output labeled as risk score, not shutoff recommendation